In [1]:
import os

import pandas as pd
import numpy as np

import regex as re
from datetime import datetime

In [2]:
data_path = "../data/in/"
output_path = "../data/out/"

regions = {"vor": "20241214-0617_gtfs_vor_2024", #vienna, lower austria, burgenland
           "ooevv": "20241212-0156_gtfs_ooevv_2024", #upper austria
           "esg": "20241203-0058_gtfs_esg_2024", #linz
           "verbundlinie": "20241217-0310_gtfs_verbundlinie_2024", #styria
           "kaernterlinien": "20241214-0253_gtfs_kaerntnerlinien_2024", #carinthia
           "salzburgverkehr": "20241217-0359_gtfs_salzburgverkehr_2024", #salzburg
           "vvt": "20241217-0436_gtfs_vvt_2024", #tyrol
           "vmobil": "20241212-0624_gtfs_vmobil_2024", #vorarlberg
           "obb": "GTFS_2024_obb"} #oebb maybe 20241217-0222_gtfs_evu_2024

# select region
state_name = "vmobil"
# select day for calculation in format YYYYMMDD in 2024
selected_day = 20240530

# stop categories
table_roman = np.array([
    ["I", "I", "II", "III"],        # < 5 min
    ["I", "II", "III", "III"],      # 5 >= x <= 10
    ["II", "III", "IV", "IV"],      # 10 < x < 20
    ["III", "IV", "V", "V"],        # 20 >= x < 40
    ["IV", "V", "VI", "VI"],        # 40 >= x <= 60
    ["V", "VI", "VII", "VII"],      # 60 < x <= 120  
    ["X", "VII", "VIII", "VIII"],    # 120 < x <= 210 
    ["X", "X", "X", "X"],               # > 210 
                                    # X = empty, i.e. worst case
])
table = np.array([
    [0, 0, 0, 0],        # < 5 min
    [0, 1, 2, 2],        # 5 >= x <= 10
    [1, 2, 3, 3],        # 10 < x < 20
    [2, 3, 4, 4],        # 20 >= x < 40
    [3, 4, 5, 5],        # 40 >= x <= 60
    [4, 5, 6, 6],        # 60 < x <= 120  
    [-1, 6, 7, 7],       # 120 < x <= 210 
    [-1, -1, -1, -1],    # > 210
])

# transport_category = ["Fernverkehr REX", 
#                       "S-Bahn / U-Bahn, Regionalbahn, Schnellbus, Lokalbahn", 
#                       "Straßenbahn, Metrobus, 0-Bus", 
#                       "Bus"]
route_type_translation = {0: 2, 1: 1, 2: 0, 3: 3, 11: 3, 7: 3, 4: 3}



In [3]:
def lookup_category(interval, t_cat):
    """
    Lookup the stop category based on the interval and the transport type.
    """
    if interval < 5:
        return table[0][t_cat]
    elif interval <= 10:
        return table[1][t_cat]
    elif interval < 20:
        return table[2][t_cat]
    elif interval < 40:
        return table[3][t_cat]
    elif interval <= 60:
        return table[4][t_cat]
    elif interval <= 120:
        return table[5][t_cat]
    elif interval <= 210:
        return table[6][t_cat]
    else:
        return table[7][t_cat]
    
def category_to_roman(t_cat, reverse=False):
    """
    Convert a category to a roman numeral or vice versa if reverse is True
    """
    roman_numerals = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "X"]
    if reverse:
        return roman_numerals.index(t_cat)
    return roman_numerals[t_cat]
    
def detect_route_type(trip_name, route_type):
    """
    Detect the type of transportation based on the trip name and the route type
    By default, the transport type is only based on the route type.
    For route type 2 (train), the trip_name is used, to further distinguish between different trains, if available.
    """
    # TODO: also consider route_short/long_name ??? 
    if route_type == 2 and not pd.isna(trip_name):
        trips_name = trip_name.lower()
        if any(x in trips_name for x in ["rj", "rjx", "nj", "en", "ic", "ec", "ice", "ecb", "rex"]): #fernverkehr
            return 0
        else:
            return 1
    else:
        return route_type_translation[route_type]

In [4]:
# read in the data
path = f"{data_path}/{regions[state_name]}/"

stops = pd.read_csv(path + "/stops.txt", quotechar='"', sep=",")
stop_times = pd.read_csv(path + "/stop_times.txt", quotechar='"', sep=",")
trips = pd.read_csv(path + "/trips.txt", quotechar='"', sep=",")
routes = pd.read_csv(path + "/routes.txt", quotechar='"', sep=",")
calendar = pd.read_csv(path + "/calendar.txt", quotechar='"', sep=",")
calendar_dates = pd.read_csv(path + "/calendar_dates.txt", quotechar='"', sep=",")

print(stops.shape)
stops.head()

(6164, 9)


,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code
0,at:47:1222:0:4,St. Anton a.A. Bahnhof,47.127468,10.266639,6455.0,NaN,Pat:47:1222,Level 0,1
1,at:47:1222:22,Steig 2+3,47.127450,10.267232,NaN,NaN,Pat:47:1222,Level 0,NaN
2,at:47:61099:0:1,St. Anton a. A. Kohlereck,47.122358,10.253820,6455.0,NaN,Pat:47:61099,Level 0,1
3,at:47:61099:0:2,St. Anton a. A. Kohlereck,47.122322,10.253703,6455.0,NaN,Pat:47:61099,Level 0,2
4,at:47:62209:0:1,St. Anton a. A. Stadle B197,47.122273,10.248646,6455.0,NaN,Pat:47:62209,Level 0,1


In [5]:
calendar_filtered = calendar[(calendar['start_date'] <= selected_day) & (calendar['end_date'] >= selected_day)]
calendar_dates_filtered = calendar_dates[calendar_dates["date"] == selected_day]

# find weekday of selected day
days = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
date = datetime.fromisoformat(str(selected_day))
day_string = days[date.weekday()]

# only keep services that run on that weekday
calendar_filtered = calendar_filtered[calendar_filtered[day_string] == 1]

print(calendar_filtered.shape)
calendar_filtered.head()

(479, 10)


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,T0,1,1,1,1,1,0,0,20231210,20241214
2,T0+02o00,1,1,1,1,1,0,0,20231210,20241214
4,T0+05p00,1,1,1,1,1,0,0,20231210,20241214
6,T0+0a710,1,1,1,1,1,0,0,20231210,20241214
7,T0+0to00,1,1,1,1,1,0,0,20231210,20241214


In [6]:
# keep only valid trips
trips_filtered = trips[trips['service_id'].isin(calendar_filtered['service_id'])]
print(trips_filtered.shape)

trips_filtered = trips_filtered[trips_filtered["service_id"].isin(calendar_dates_filtered[calendar_dates_filtered["exception_type"] == 2]["service_id"])]
print(trips_filtered.shape)

trips_full = pd.concat([trips_filtered, trips[trips["service_id"].isin(calendar_dates_filtered[calendar_dates_filtered["exception_type"]==1]["service_id"])]])
print(trips_full.shape)
trips_full.head()

(12968, 8)
(11597, 8)
(13325, 8)


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
0,at:vvv:101:,T0+qcp00,1.T0.31-101-E-j24-1.1.H,31-101-E-j24-1.1.H,Bregenz Pfänderbahn,NaN,0,NaN
3,at:vvv:101:,T0,10.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
5,at:vvv:101:,T0,11.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
8,at:vvv:101:,T0,12.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
10,at:vvv:101:,T0,13.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN


In [7]:
# merge trips with routes information
routes_trips = pd.merge(trips_full, routes, on='route_id', how='left')

# translate route type
routes_trips['trip_short_name'] = routes_trips['trip_short_name'].astype('str')
routes_trips['rank'] = routes_trips.apply(lambda x: detect_route_type(x['trip_short_name'], x['route_type']), axis=1)

print(routes_trips.shape)
routes_trips[routes_trips['rank'].isna()]

(13325, 13)


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id,agency_id,route_short_name,route_long_name,route_type,rank


In [8]:
# prepare stops and stop_times
stops_filtered = stops.copy()
stops_filtered['stop_id'] = stops_filtered['stop_id'].astype(str).apply(
    lambda x: (
        re.match(r'^((?:[^:]*:){3})', x).group(1).rstrip(':')
        if re.match(r'^((?:[^:]*:){3})', x)
        else x
    )
)
# keep only stop entry for parent station, if no parent station is given, keep a stop entry
stops_parents = stops_filtered[stops_filtered['stop_id'].str.startswith('Pat')].copy()
stops_parents['stop_id'] = stops_parents['stop_id'].apply(lambda x: x if x[0] != 'P' else x[1:])
stops_filtered = stops_filtered[stops_filtered['stop_id'].str.startswith('at')]
stops_filtered = stops_filtered[~stops_filtered['stop_id'].isin(stops_parents['stop_id'])].drop_duplicates(subset=['stop_id'], keep='first')
# TODO: maybe filter out special stations e.g. obb_CP_80854 Wattens Sammelpunkt Bahnhofstraße MPREIS or Pat:42:99979_HoB

stops_filtered_final = pd.concat([stops_parents, stops_filtered])
print(stops_filtered_final.shape)

stop_times_filtered = stop_times.copy()
stop_times_filtered = stop_times_filtered[stop_times_filtered['departure_time'].between('06:00:00', '20:00:00')]
stop_times_filtered = stop_times_filtered[stop_times_filtered['stop_id'].str.startswith('at')]
# TODO: check if Parent station is in stop_times and keep those
stop_times_filtered['stop_id'] = stop_times_filtered['stop_id'].astype(str).apply(
    lambda x: (
        re.match(r'^((?:[^:]*:){3})', x).group(1).rstrip(':')
        if re.match(r'^((?:[^:]*:){3})', x)
        else x
    )
)

# display(stop_times_df_obb_mod.head())
stops_filtered_final

(1766, 9)


,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code
4118,at:47:1222,St. Anton am Arlberg Bahnhof,47.127389,10.266782,NaN,1.0,NaN,NaN,NaN
4119,at:47:61099,St. Anton a. A. Kohlereck,47.122346,10.253766,NaN,1.0,NaN,NaN,NaN
4120,at:47:62209,St. Anton a. A. Stadle B197,47.122267,10.248583,NaN,1.0,NaN,NaN,NaN
4121,at:47:62504,St. Anton a. A. Brandli,47.124901,10.260108,NaN,1.0,NaN,NaN,NaN
4122,at:47:64938,St. Anton a. A. Terminal West,47.126527,10.263333,NaN,1.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
888,at:48:1436,Rankweil Bahnhof,47.271513,9.637944,3732.0,NaN,NaN,Level 0,A
2460,at:48:356,Wolfurt Bahnhof,47.455283,9.741789,2711.0,NaN,NaN,Level 0,1
2523,at:48:394,Altach Bahnhof,47.352579,9.661255,3909.0,NaN,NaN,Level 0,5
3016,at:48:628,Dornbirn Hatlerdorf Bahnhof,47.397249,9.725763,2020.0,NaN,NaN,Level 0,1


In [9]:
stop_times_trips = pd.merge(stop_times_filtered, routes_trips, on='trip_id', how='inner')
print(stop_times_trips.shape)
stop_times_trips.head()

(219948, 21)


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,route_id,...,shape_id,trip_headsign,trip_short_name,direction_id,block_id,agency_id,route_short_name,route_long_name,route_type,rank
0,1.T0.12-820-E-j24-1.20.R,06:00:00,06:00:00,at:48:1232,28,NaN,0,0,19261.74,at:vvv:820:,...,12-820-E-j24-1.20.R,Bregenz Bahnhof,nan,1,NaN,46.0,820,820,3,3
1,1.T0.12-820-E-j24-1.20.R,06:01:00,06:01:00,at:48:1233,29,NaN,0,0,20149.77,at:vvv:820:,...,12-820-E-j24-1.20.R,Bregenz Bahnhof,nan,1,NaN,46.0,820,820,3,3
2,1.T0.12-820-E-j24-1.20.R,06:02:00,06:02:00,at:48:1228,30,NaN,0,0,20924.20,at:vvv:820:,...,12-820-E-j24-1.20.R,Bregenz Bahnhof,nan,1,NaN,46.0,820,820,3,3
3,1.T0.12-820-E-j24-1.20.R,06:03:00,06:03:00,at:48:1229,31,NaN,0,0,21372.79,at:vvv:820:,...,12-820-E-j24-1.20.R,Bregenz Bahnhof,nan,1,NaN,46.0,820,820,3,3
4,1.T0.12-820-E-j24-1.20.R,06:05:00,06:05:00,at:48:1235,32,NaN,0,0,22450.46,at:vvv:820:,...,12-820-E-j24-1.20.R,Bregenz Bahnhof,nan,1,NaN,46.0,820,820,3,3


In [10]:
stop_times_grouped = stop_times_trips.groupby(['stop_id']).agg(rank=("rank", "min"), count=("rank", "count")).reset_index()

# TODO: maybe after merge with obb stations
stop_times_grouped["interval"] = stop_times_grouped["count"].apply(lambda x: 840 / (x/2))
stop_times_grouped["category"] = stop_times_grouped.apply(lambda x: lookup_category(x["interval"], x["rank"]), axis=1)

stop_times_grouped

,stop_id,rank,count,interval,category
0,at:47:1222,3,71,23.661972,4
1,at:47:61099,3,71,23.661972,4
2,at:47:62209,3,36,46.666667,5
3,at:47:62504,3,36,46.666667,5
4,at:47:64938,3,36,46.666667,5
...,...,...,...,...,...
1718,at:48:992,3,125,13.440000,3
1719,at:48:993,3,57,29.473684,4
1720,at:48:995,3,130,12.923077,3
1721,at:48:996,3,120,14.000000,3


In [11]:
# TODO: change "how" to "left" to include all stops, also ones without rank and count (nan)
stops_final = pd.merge(stops_filtered_final.drop(['zone_id', 'location_type', 'level_id', 'platform_code', 'parent_station'], axis=1), stop_times_grouped, on='stop_id', how='inner')
stops_final

,stop_id,stop_name,stop_lat,stop_lon,rank,count,interval,category
0,at:47:1222,St. Anton am Arlberg Bahnhof,47.127389,10.266782,3,71,23.661972,4
1,at:47:61099,St. Anton a. A. Kohlereck,47.122346,10.253766,3,71,23.661972,4
2,at:47:62209,St. Anton a. A. Stadle B197,47.122267,10.248583,3,36,46.666667,5
3,at:47:62504,St. Anton a. A. Brandli,47.124901,10.260108,3,36,46.666667,5
4,at:47:64938,St. Anton a. A. Terminal West,47.126527,10.263333,3,36,46.666667,5
...,...,...,...,...,...,...,...,...
1718,at:48:1436,Rankweil Bahnhof,47.271513,9.637944,3,593,2.833052,0
1719,at:48:356,Wolfurt Bahnhof,47.455283,9.741789,3,76,22.105263,4
1720,at:48:394,Altach Bahnhof,47.352579,9.661255,3,58,28.965517,4
1721,at:48:628,Dornbirn Hatlerdorf Bahnhof,47.397249,9.725763,3,123,13.658537,3


In [12]:
# save to file
# stops_final.to_csv(output_path + f"stops_{state_name}_{selected_day}.csv", index=False)